# S19 — Fine-Tuning & LoRA

**Week 10 · Wed Oct 28, 2026 · Module 3**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Boyu-Zhang-UOI/dl-f2026-notebooks/blob/main/s19_fine_tuning_lora.ipynb)

Every cell below is a worked example from the [S19 reading](https://boyu-zhang-uoi.github.io/dl-f2026/readings/sessions/s19/) — same code, same seeds, same outputs. Run them, then change things and see what breaks: that is what this notebook is for.

Slides for this session: [s19.html](https://boyu-zhang-uoi.github.io/dl-f2026/slides/s19.html)


In [ ]:
# Colab only: install PyTorch if it is missing (local runs already have it).
try:
    import torch  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch"], check=True)
    import torch  # noqa: F401
print("environment ready")

## The cost of full fine-tuning


*Expected output starts with:* `               model   full FT (GB)   LoRA r=8 ~0.1% (GB)`


In [ ]:
# Memory arithmetic for full fine-tuning with Adam/AdamW.
# Per trainable parameter, a standard fp32 setup stores:
#   weight (4 bytes) + gradient (4) + Adam m (4) + Adam v (4) = 16 bytes.
# Common mixed-precision setup (bf16 weights/grads, fp32 master + states):
#   bf16 weight (2) + bf16 grad (2) + fp32 master copy (4) + m (4) + v (4) = 16 bytes.
# Activations come on top of this and depend on batch/sequence length.

GB = 1024 ** 3

def full_ft_memory(n_params, bytes_per_param=16):
    return n_params * bytes_per_param / GB

def lora_memory(n_params, n_trainable, weight_bytes=2):
    # frozen base weights (no grads, no optimizer states) + full stack for LoRA params
    return (n_params * weight_bytes + n_trainable * 16) / GB

models = [("124M (GPT-2 small)", 124e6),
          ("1.3B", 1.3e9),
          ("7B", 7e9),
          ("70B", 70e9)]

print(f"{'model':>20} {'full FT (GB)':>14} {'LoRA r=8 ~0.1% (GB)':>21}")
for name, n in models:
    n_lora = 0.001 * n  # ~0.1% trainable, a typical LoRA budget
    print(f"{name:>20} {full_ft_memory(n):>14.1f} {lora_memory(n, n_lora):>21.1f}")

n = 7e9
print(f"\nbreakdown for the 7B model, full fine-tuning (fp32):")
print(f"  weights            : {n * 4 / GB:6.1f} GB")
print(f"  gradients          : {n * 4 / GB:6.1f} GB")
print(f"  Adam m (momentum)  : {n * 4 / GB:6.1f} GB")
print(f"  Adam v (variance)  : {n * 4 / GB:6.1f} GB")
print(f"  total (no activations): {full_ft_memory(n):.1f} GB")

## LoRA from scratch


*Expected output starts with:* `total params    : 10,930`


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

class LoRALinear(nn.Module):
    """A frozen nn.Linear plus a trainable low-rank update (alpha/r) * B @ A."""
    def __init__(self, base: nn.Linear, r=4, alpha=8):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad = False          # freeze W and bias
        self.r, self.alpha = r, alpha
        self.A = nn.Parameter(torch.randn(r, base.in_features) * 0.01)
        self.B = nn.Parameter(torch.zeros(base.out_features, r))

    def forward(self, x):
        return self.base(x) + (self.alpha / self.r) * (x @ self.A.T @ self.B.T)

    def merged_weight(self):
        return self.base.weight + (self.alpha / self.r) * (self.B @ self.A)

# --- A small "pretrained" model, then adapt it to a new task with LoRA ------
d_in, d_hidden, d_out = 64, 128, 10
model = nn.Sequential(nn.Linear(d_in, d_hidden), nn.ReLU(),
                      nn.Linear(d_hidden, d_out))
# pretend this model is pretrained; wrap both Linears with LoRA
model[0] = LoRALinear(model[0], r=4, alpha=8)
model[2] = LoRALinear(model[2], r=4, alpha=8)

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"total params    : {total:,}")
print(f"trainable (LoRA): {trainable:,} ({100 * trainable / total:.2f}%)")

# synthetic classification task
g = torch.Generator().manual_seed(1)
X = torch.randn(512, d_in, generator=g)
w_true = torch.randn(d_in, d_out, generator=g)
y = (X @ w_true).argmax(dim=1)

opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-2)
for step in range(201):
    loss = F.cross_entropy(model(X), y)
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 100 == 0:
        acc = (model(X).argmax(1) == y).float().mean()
        print(f"step {step:3d}  loss {loss.item():.4f}  acc {acc.item():.4f}")

# --- Merge check: replacing base weight with W + (alpha/r) B @ A ------------
with torch.no_grad():
    merged = nn.Sequential(nn.Linear(d_in, d_hidden), nn.ReLU(),
                           nn.Linear(d_hidden, d_out))
    for i in (0, 2):
        merged[i].weight.copy_(model[i].merged_weight())
        merged[i].bias.copy_(model[i].base.bias)
    x_test = torch.randn(64, d_in, generator=g)
    diff = (model(x_test) - merged(x_test)).abs().max()
print(f"max |LoRA model - merged model| on fresh inputs: {diff.item():.2e}")

## Three methods, one task pair


*Expected output starts with:* `entropy floor of either language: 0.4349 nats/token`


In [ ]:
# Three ways to adapt one pretrained model: full FT, LoRA, and prompt tuning.
# Pretrain a tiny causal LM on language A (a forward cycle over 4 symbols),
# then adapt it to language B (the reversed cycle) with each method.
import copy
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

V, CTX, D = 4, 16, 32

def make_chain(p_follow, cycle, n_seq, gen):
    """Markov chains over {0..3}: follow the cycle w.p. p_follow, else uniform."""
    T = torch.full((V, V), (1 - p_follow) / 3)
    for s in range(V):
        T[s, cycle[s]] = p_follow
    seqs = torch.zeros(n_seq, CTX + 1, dtype=torch.long)
    seqs[:, 0] = torch.randint(0, V, (n_seq,), generator=gen)
    for t in range(CTX):
        seqs[:, t + 1] = torch.multinomial(T[seqs[:, t]], 1, generator=gen).squeeze(1)
    return seqs

gen = torch.Generator().manual_seed(1)
task_A = make_chain(0.9, [1, 2, 3, 0], 2000, gen)   # a->b->c->d->a
task_B = make_chain(0.9, [3, 0, 1, 2], 2000, gen)   # a->d->c->b->a
H = -(0.9 * math.log(0.9) + 0.1 * math.log(0.1 / 3))
print(f"entropy floor of either language: {H:.4f} nats/token")

class TinyLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(V, D)
        self.pos = nn.Embedding(CTX, D)
        self.ln1, self.ln2 = nn.LayerNorm(D), nn.LayerNorm(D)
        self.q, self.k, self.v, self.o = (nn.Linear(D, D) for _ in range(4))
        self.up, self.down = nn.Linear(D, 4 * D), nn.Linear(4 * D, D)
        self.head = nn.Linear(D, V)

    def forward(self, idx, prompt=None):
        x = self.emb(idx) + self.pos(torch.arange(idx.size(1)))
        n_p = 0
        if prompt is not None:                     # prompt tuning: prepend
            n_p = prompt.size(1)                   # learned "virtual tokens"
            x = torch.cat([prompt.expand(x.size(0), -1, -1), x], dim=1)
        T = x.size(1)
        h = self.ln1(x)
        q, k, v = self.q(h), self.k(h), self.v(h)
        att = (q @ k.transpose(1, 2)) / math.sqrt(D)
        att = att.masked_fill(torch.ones(T, T).tril() == 0, float("-inf"))
        x = x + self.o(att.softmax(-1) @ v)
        x = x + self.down(F.gelu(self.up(self.ln2(x))))
        return self.head(x)[:, n_p:, :]            # logits for real positions

class LoRALinear(nn.Module):
    def __init__(self, base, r=4, alpha=8):
        super().__init__()
        self.base, self.r, self.alpha = base, r, alpha
        for p in self.base.parameters():
            p.requires_grad = False
        self.A = nn.Parameter(torch.randn(r, base.in_features) * 0.01)
        self.B = nn.Parameter(torch.zeros(base.out_features, r))

    def forward(self, x):
        return self.base(x) + (self.alpha / self.r) * (x @ self.A.T @ self.B.T)

def lm_loss(model, seqs, **kw):
    logits = model(seqs[:, :-1], **kw)
    return F.cross_entropy(logits.reshape(-1, V), seqs[:, 1:].reshape(-1))

def train(model, seqs, params, steps=300, lr=3e-3, **kw):
    opt = torch.optim.AdamW(params, lr=lr)
    for step in range(steps):
        loss = lm_loss(model, seqs[torch.randint(0, len(seqs), (64,))], **kw)
        opt.zero_grad(); loss.backward(); opt.step()
    return model

# --- pretrain on task A ----------------------------------------------------
base = TinyLM()
train(base, task_A[:1600], base.parameters())
with torch.no_grad():
    print(f"after pretraining: loss on A = {lm_loss(base, task_A[1600:]).item():.4f}")

# --- adapt to task B three ways -------------------------------------------
results = []
with torch.no_grad():
    results.append(("no adaptation", 0, lm_loss(base, task_B[1600:]).item()))

full = copy.deepcopy(base)
train(full, task_B[:1600], full.parameters())
with torch.no_grad():
    results.append(("full fine-tune", sum(p.numel() for p in full.parameters()),
                    lm_loss(full, task_B[1600:]).item()))

lora = copy.deepcopy(base)
for p in lora.parameters():
    p.requires_grad = False
lora.q, lora.v = LoRALinear(lora.q), LoRALinear(lora.v)
lora_params = [p for p in lora.parameters() if p.requires_grad]
train(lora, task_B[:1600], lora_params)
with torch.no_grad():
    results.append(("LoRA (r=4, q+v)", sum(p.numel() for p in lora_params),
                    lm_loss(lora, task_B[1600:]).item()))

pt = copy.deepcopy(base)
for p in pt.parameters():
    p.requires_grad = False
prompt = nn.Parameter(pt.emb.weight.detach().mean(0, keepdim=True)
                      .repeat(8, 1).unsqueeze(0).clone())
train(pt, task_B[:1600], [prompt], lr=3e-2, prompt=prompt)
with torch.no_grad():
    results.append(("prompt tuning (8 tok)", prompt.numel(),
                    lm_loss(pt, task_B[1600:], prompt=prompt).item()))

print(f"\n{'method':>22} {'trainable':>10} {'loss on B':>10}")
for name, n, loss in results:
    print(f"{name:>22} {n:>10,} {loss:>10.4f}")

## Catastrophic forgetting, demonstrated


*Expected output starts with:* `                             model    loss A    loss B   (floor 0.4349)`


In [ ]:
# Catastrophic forgetting on a task pair, and two ways out.
# Pretrain on language A, fine-tune on language B, and watch what happens
# to task A. Then: (1) replay a little A data, (2) use a removable adapter.
import copy
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

V, CTX, D = 4, 16, 32

def make_chain(p_follow, cycle, n_seq, gen):
    T = torch.full((V, V), (1 - p_follow) / 3)
    for s in range(V):
        T[s, cycle[s]] = p_follow
    seqs = torch.zeros(n_seq, CTX + 1, dtype=torch.long)
    seqs[:, 0] = torch.randint(0, V, (n_seq,), generator=gen)
    for t in range(CTX):
        seqs[:, t + 1] = torch.multinomial(T[seqs[:, t]], 1, generator=gen).squeeze(1)
    return seqs

gen = torch.Generator().manual_seed(1)
task_A = make_chain(0.9, [1, 2, 3, 0], 2000, gen)
task_B = make_chain(0.9, [3, 0, 1, 2], 2000, gen)

class TinyLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(V, D)
        self.pos = nn.Embedding(CTX, D)
        self.ln1, self.ln2 = nn.LayerNorm(D), nn.LayerNorm(D)
        self.q, self.k, self.v, self.o = (nn.Linear(D, D) for _ in range(4))
        self.up, self.down = nn.Linear(D, 4 * D), nn.Linear(4 * D, D)
        self.head = nn.Linear(D, V)

    def forward(self, idx):
        x = self.emb(idx) + self.pos(torch.arange(idx.size(1)))
        T = x.size(1)
        h = self.ln1(x)
        q, k, v = self.q(h), self.k(h), self.v(h)
        att = (q @ k.transpose(1, 2)) / math.sqrt(D)
        att = att.masked_fill(torch.ones(T, T).tril() == 0, float("-inf"))
        x = x + self.o(att.softmax(-1) @ v)
        x = x + self.down(F.gelu(self.up(self.ln2(x))))
        return self.head(x)

class LoRALinear(nn.Module):
    def __init__(self, base, r=4, alpha=8):
        super().__init__()
        self.base, self.r, self.alpha = base, r, alpha
        for p in self.base.parameters():
            p.requires_grad = False
        self.A = nn.Parameter(torch.randn(r, base.in_features) * 0.01)
        self.B = nn.Parameter(torch.zeros(base.out_features, r))

    def forward(self, x):
        return self.base(x) + (self.alpha / self.r) * (x @ self.A.T @ self.B.T)

def lm_loss(model, seqs):
    logits = model(seqs[:, :-1])
    return F.cross_entropy(logits.reshape(-1, V), seqs[:, 1:].reshape(-1))

def train(model, seqs, params, steps=300, lr=3e-3):
    opt = torch.optim.AdamW(params, lr=lr)
    for step in range(steps):
        loss = lm_loss(model, seqs[torch.randint(0, len(seqs), (64,))])
        opt.zero_grad(); loss.backward(); opt.step()
    return model

def report(name, model):
    with torch.no_grad():
        la = lm_loss(model, task_A[1600:]).item()
        lb = lm_loss(model, task_B[1600:]).item()
    print(f"{name:>34} {la:>9.4f} {lb:>9.4f}")

base = TinyLM()
train(base, task_A[:1600], base.parameters())

print(f"{'model':>34} {'loss A':>9} {'loss B':>9}   (floor 0.4349)")
report("pretrained on A", base)

# 1) plain full fine-tuning on B: learns B, forgets A
full = copy.deepcopy(base)
train(full, task_B[:1600], full.parameters())
report("full FT on B", full)

# 2) full fine-tuning on B with 10% replay of A data
replay = copy.deepcopy(base)
mixed = torch.cat([task_B[:1600], task_A[:178]])   # ~10% A
train(replay, mixed, replay.parameters())
report("full FT on B + 10% A replay", replay)

# 3) LoRA adapter on B: the base weights never move
lora = copy.deepcopy(base)
for p in lora.parameters():
    p.requires_grad = False
q_base, v_base = lora.q, lora.v                    # keep handles to the originals
lora.q, lora.v = LoRALinear(q_base), LoRALinear(v_base)
train(lora, task_B[:1600], [p for p in lora.parameters() if p.requires_grad])
report("LoRA on B (adapter attached)", lora)

lora.q, lora.v = q_base, v_base                    # swap the adapter back out
report("LoRA on B (adapter removed)", lora)

## QLoRA: quantizing the frozen base


*Expected output starts with:* ` model   full FT    LoRA   QLoRA   fits on (QLoRA)`


In [ ]:
# QLoRA memory arithmetic: what does quantizing the frozen base buy?
# Full FT (fp32/mixed): 16 bytes per parameter in weights+grads+Adam states.
# LoRA:  frozen bf16 base (2 bytes/param) + 16 bytes per LoRA parameter.
# QLoRA: frozen 4-bit base (0.5 bytes/param, NF4) + ~0.03 bits/param of
#        block-wise scale factors (double-quantized) + 16 bytes per LoRA param.

GB = 1024 ** 3

def full_ft(n):
    return 16 * n / GB

def lora(n, frac=0.001):
    return (2 * n + 16 * frac * n) / GB

def qlora(n, frac=0.001):
    base = 0.5 * n            # NF4: 4 bits per weight
    scales = n / 64 * 1       # one 8-bit quantized scale per 64-weight block
    return (base + scales + 16 * frac * n) / GB

gpus = [("RTX 4090", 24), ("A6000", 48), ("A100/H100", 80)]

print(f"{'model':>6} {'full FT':>9} {'LoRA':>7} {'QLoRA':>7}   fits on (QLoRA)")
for name, n in [("1.3B", 1.3e9), ("7B", 7e9), ("13B", 13e9),
                ("33B", 33e9), ("65B", 65e9)]:
    q = qlora(n)
    fits = ", ".join(g for g, mem in gpus if q < 0.85 * mem)  # leave activation room
    print(f"{name:>6} {full_ft(n):>8.1f}G {lora(n):>6.1f}G {q:>6.1f}G   {fits or '(none)'}")

n = 65e9
print(f"\n65B breakdown under QLoRA:")
print(f"  4-bit frozen base        : {0.5 * n / GB:6.1f} GB")
print(f"  quantization scales      : {n / 64 / GB:6.1f} GB")
print(f"  LoRA params + grads+Adam : {16 * 0.001 * n / GB:6.1f} GB")
print(f"  total (no activations)   : {qlora(n):6.1f} GB")

## Try it yourself

1. Extend the memory script with an activation estimate for a Transformer: assume `2 * n_layers * seq_len * batch * d_model` bf16 values must be stored for backprop, and plot how the full-fine-tuning total for the 7B model (`n_layers = 32`, `d_model = 4096`) grows with sequence length from 512 to 8192.
2. In the LoRA script, sweep `r` over 1, 2, 4, 8, 16 and record final loss and trainable-parameter count. Where does this task stop benefiting from rank?
3. Verify the *unmerge* property: after merging, subtract `(alpha/r) * B @ A` from the merged weight and confirm you recover the original base model's outputs on test inputs, reporting the maximum absolute difference.
4. Wrap only the first layer with LoRA (leave the second frozen without any adapter) and retrain. Can a rank-4 update of one layer still solve the task? What does that suggest about where adaptation capacity is needed?


---

Full discussion of everything above: [S19 reading](https://boyu-zhang-uoi.github.io/dl-f2026/readings/sessions/s19/).
